# Day 03: Embeddings — From Integers to Vectors(Day 03:嵌入——从整数到向量)

**Time:** ~20 minutes  
**Prerequisites:** Day 02 (What's Inside a Model, Tokenization)  
**Goal:** Understand how token IDs become the 768-dimensional vectors the model works with.

> **时长:**约 20 分钟  
> **前置要求:**Day 02(模型里面有什么、分词)  
> **目标:**理解词元 ID 是如何变成模型实际处理的 768 维向量的。

---

So far: we can parse model weights (02a) and tokenize text into IDs (02b).  
But the model doesn't do math on integers. It needs **vectors** — dense arrays of floating-point numbers.

> 到目前为止:我们已经能解析模型权重(02a)、把文本分词成 ID(02b)。  
> 但模型并不对整数做数学运算,它需要的是**向量(vector)**——稠密的浮点数数组。

Embeddings are the bridge: `integer ID -> 768-dim vector`.

> 嵌入(embedding)就是这座桥:`整数 ID -> 768 维向量`。

Think of it like DNS: a domain name (`cat`) is human-readable but useless to a router.  
DNS resolves it to an IP address (`192.168.1.42`) that the network can route.  
Embeddings resolve a token ID (`9246`) to a vector that the neural network can route through its layers.

> 可以把它类比成 DNS:域名(`cat`)人类可读,但对路由器没用;  
> DNS 把它解析成网络可以路由的 IP 地址(`192.168.1.42`);  
> 嵌入则把词元 ID(`9246`)“解析”成一个向量,让神经网络能够在层层网络中路由它。

## Setup: Load Weights + Tokenizer from Previous Notebooks(准备:加载前几个 notebook 的权重和分词器)

We need the model weights and the tokenizer. Same code as 02a and 02b — copy-pasted for self-containment.

> 我们需要模型权重和分词器。代码与 02a、02b 相同——直接复制过来,保证本 notebook 自包含。

In [2]:
import json
import struct
import os
import re
import numpy as np

MODEL_DIR = "../day02/gpt2_weights"

# --- Load weights (from 02a) ---(加载权重,来自 02a)
def parse_safetensors(path):
    with open(path, "rb") as f:
        header_len = struct.unpack("<Q", f.read(8))[0]
        header = json.loads(f.read(header_len))
        data_start = 8 + header_len
        tensors = {}
        for name, info in header.items():
            if name == "__metadata__":
                continue
            dtype = {"F32": np.float32, "F16": np.float16}[info["dtype"]]
            start, end = info["data_offsets"]
            f.seek(data_start + start)
            tensors[name] = np.frombuffer(f.read(end - start), dtype=dtype).reshape(info["shape"])
        return tensors

weights = parse_safetensors(os.path.join(MODEL_DIR, "model.safetensors"))

# --- Load tokenizer (from 02b) ---(加载分词器,来自 02b)
with open(os.path.join(MODEL_DIR, "vocab.json")) as f:
    vocab = json.load(f)
with open(os.path.join(MODEL_DIR, "merges.txt")) as f:
    merges_raw = f.read().strip().split("\n")[1:]

id_to_token = {v: k for k, v in vocab.items()}
bpe_ranks = {tuple(m.split()): i for i, m in enumerate(merges_raw)}

def bytes_to_unicode():
    bs = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return dict(zip(bs, [chr(c) for c in cs]))

byte_encoder = bytes_to_unicode()
byte_decoder = {v: k for k, v in byte_encoder.items()}
GPT2_PATTERN = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\w+| ?[^\s\w]+|\s+(?!\S)|\s+""")

def get_pairs(tokens):
    return set(zip(tokens[:-1], tokens[1:]))

def bpe_merge(tokens):
    while True:
        pairs = get_pairs(tokens)
        if not pairs:
            break
        best = min(pairs, key=lambda p: bpe_ranks.get(p, float("inf")))
        if best not in bpe_ranks:
            break
        first, second = best
        new = []
        i = 0
        while i < len(tokens):
            if i < len(tokens)-1 and tokens[i] == first and tokens[i+1] == second:
                new.append(first + second)
                i += 2
            else:
                new.append(tokens[i])
                i += 1
        tokens = new
    return tokens

def encode(text):
    ids = []
    for chunk in re.findall(GPT2_PATTERN, text):
        uc = [byte_encoder[b] for b in chunk.encode("utf-8")]
        for t in bpe_merge(uc):
            ids.append(vocab[t])
    return ids

def decode(token_ids):
    text = "".join(id_to_token[i] for i in token_ids)
    return bytes([byte_decoder[c] for c in text]).decode("utf-8", errors="replace")

print(f"Weights loaded: {len(weights)} tensors")
print(f"Tokenizer ready: {len(vocab):,} tokens")

Weights loaded: 160 tensors
Tokenizer ready: 50,257 tokens


## Step 1: Token Embeddings — The Lookup Table(第一步:词元嵌入——一张查找表)

The token embedding matrix `wte.weight` has shape `[50257, 768]`.

> 词元嵌入矩阵 `wte.weight` 的形状是 `[50257, 768]`。

- **50,257 rows** — one for every token in the vocabulary
- **768 columns** — the embedding dimension (d_model)

> - **50,257 行**——词表中每个词元各占一行
> - **768 列**——嵌入维度(d_model)

To embed a token, you just **index into this matrix**. Token ID 9246 → row 9246.  
No multiplication needed — it's a pure table lookup, like indexing into an array.

> 要嵌入一个词元,只需要**在这张矩阵里做索引**。词元 ID 9246 → 第 9246 行。  
> 不需要任何乘法——纯粹是查表,就像数组下标访问。

```
wte.weight:   50257 rows x 768 columns

           col0    col1    col2    ...   col767
row 0    [ 0.012, -0.023,  0.045, ...,  0.008 ]   <- token 0
row 1    [-0.034,  0.011, -0.007, ...,  0.015 ]   <- token 1
...
row 9246 [ 0.041, -0.019,  0.033, ..., -0.028 ]   <- "cat"
...
row 50256[ 0.003,  0.001, -0.002, ...,  0.001 ]   <- <|endoftext|>
```

> (图示:wte.weight 共 50,257 行 × 768 列;第 0 行对应词元 0,第 9246 行对应 "cat",第 50256 行对应 <|endoftext|>;嵌入 = 直接取该词元对应的那一行)

Think of it like a hash map: `embedding = wte[token_id]`

> 把它想成一个哈希表:`embedding = wte[token_id]`

**What do the 768 numbers in a row mean?**

> **一行里的 768 个数字是什么含义?**

Nobody hand-designed them — they were learned during training via gradient descent.
Through training, dimensions self-organize into features: some correlate with noun-ness vs verb-ness,
others with sentiment, formality, or topic. Most dimensions encode **mixtures** of features, not clean labels.

> 没有人手工设计它们——它们是训练时通过梯度下降学出来的。
> 经过训练,各维度会自组织成各种特征:有的与“名词性/动词性”相关,
> 有的与情感、正式程度或主题相关。大多数维度编码的是多种特征的**混合**,而不是干净的标签。

The individual values aren't interpretable on their own — what matters is the **pattern across all 768 dimensions together**.
Think of GPS coordinates: `37.77` alone means nothing, but `(37.77, -122.42)` puts you in San Francisco,
and you can measure it's closer to Oakland than New York. Similarly, tokens that behave alike in language
end up at nearby points in this 768-dimensional space.

> 单个数值孤立地看并没有可解释的含义——重要的是**全部 768 个维度合在一起的整体模式**。
> 就像 GPS 坐标:单独一个 `37.77` 毫无意义,但 `(37.77, -122.42)` 就把你定位到旧金山,
> 还能量出它离奥克兰比离纽约更近。同理,在语言中行为相似的词元,
> 最终会落在 768 维空间中彼此相近的位置。

In [3]:
wte = weights["wte.weight"]  # Token embedding table(词元嵌入表)

print(f"Token embedding matrix (wte.weight)")
print(f"  Shape: {wte.shape}")
print(f"  Dtype: {wte.dtype}")
print(f"  Size:  {wte.nbytes / (1024*1024):.1f} MB")
print()

# Look up embeddings for a sentence(查一个句子里各词元的嵌入)
text = "The cat sat"
token_ids = encode(text)

print(f"Text: '{text}'")
print(f"Token IDs: {token_ids}")
print()

for tid in token_ids:
    embedding = wte[tid]  # Just array indexing!(纯粹是数组下标!)
    token_str = decode([tid])
    print(f"  Token '{token_str}' (ID {tid}):")
    print(f"    Vector: [{embedding[0]:.4f}, {embedding[1]:.4f}, {embedding[2]:.4f}, ..., {embedding[-1]:.4f}]")
    print(f"    Shape:  {embedding.shape}")
    print(f"    Norm:   {np.linalg.norm(embedding):.4f}")

Token embedding matrix (wte.weight)
  Shape: (50257, 768)
  Dtype: float32
  Size:  147.2 MB

Text: 'The cat sat'
Token IDs: [464, 3797, 3332]

  Token 'The' (ID 464):
    Vector: [-0.0686, -0.0203, 0.0645, ..., 0.0031]
    Shape:  (768,)
    Norm:   2.7203
  Token ' cat' (ID 3797):
    Vector: [0.0100, 0.0366, 0.1640, ..., -0.0695]
    Shape:  (768,)
    Norm:   3.2578
  Token ' sat' (ID 3332):
    Vector: [0.0207, -0.1156, 0.0208, ..., -0.0454]
    Shape:  (768,)
    Norm:   3.2772


## Step 2: Position Embeddings — Where Are You in the Sequence?(第二步:位置嵌入——你在序列的哪个位置?)

Token embeddings tell the model **what** a token is, but not **where** it appears.

> 词元嵌入告诉模型一个词元**是什么**,但不能告诉模型它**出现在哪**。

**Why this matters:** addition is commutative. If you just add up token embeddings for
"dog bites man" and "man bites dog", you get the same result. The model has no idea which word came first.

> **为什么这很重要:**加法满足交换律。如果你只是把 "dog bites man" 和 "man bites dog"
> 的词元嵌入分别加起来,会得到相同的结果。模型根本分不清哪个词在前。

Position embeddings fix this. `wpe.weight` has shape `[1024, 768]`:
- **1024 rows** — one for each possible position (GPT-2's max sequence length)
- **768 columns** — same dimension as token embeddings (so they can be added)

> 位置嵌入(position embedding)解决了这个问题。`wpe.weight` 的形状是 `[1024, 768]`:
> - **1024 行**——每个可能的位置各占一行(GPT-2 的最大序列长度)
> - **768 列**——与词元嵌入同维度(这样才能相加)

**Where does the position come from?** Just the index in the token list. You tokenize the text,
and whatever order the tokens come out in, that's their position. First token → position 0, second → position 1, etc.

> **位置从哪来?**就是词元列表里的下标。对文本分词之后,
> 词元输出的先后顺序就是它们的位置:第 1 个词元 → 位置 0,第 2 个 → 位置 1,以此类推。

```python
token_ids = encode("The cat sat")  # [464, 9246, 3332]
# index 0 → "The" → wpe[0]
# index 1 → "cat" → wpe[1]
# index 2 → "sat" → wpe[2]
```

> (下标 0 → "The" → wpe[0];下标 1 → "cat" → wpe[1];下标 2 → "sat" → wpe[2])

**Where do the values come from?** Same place as `wte` — `wpe.weight` is a learned weight matrix
stored in the model file. Gradient descent adjusted every value during training. Nobody hand-coded
what each position vector should look like.

> **这些数值从哪来?**和 `wte` 一样——`wpe.weight` 也是存在模型文件里的、学习出来的权重矩阵。
> 训练时,梯度下降调整了其中的每个值。没有任何人手工规定过
> 每个位置向量应该长什么样。

1024 rows = GPT-2's max sequence length. It literally can't process more than 1024 tokens
because there's no position embedding for position 1024+.

> 1024 行 = GPT-2 的最大序列长度。它真的无法处理超过 1024 个词元,
> 因为位置 1024 及之后根本没有对应的位置嵌入。

The final input to the model is: `token_embedding + position_embedding`

> 模型的最终输入是:`token_embedding + position_embedding`(词元嵌入 + 位置嵌入)

```
 Position 0: wte["The"]  + wpe[0]  = input vector for "The" at position 0
 Position 1: wte["cat"]  + wpe[1]  = input vector for "cat" at position 1
 Position 2: wte["sat"]  + wpe[2]  = input vector for "sat" at position 2
```

> (位置 0:wte["The"] + wpe[0] = "The" 在位置 0 的输入向量;位置 1、2 同理)

**Why does addition work?** The token vector says **what** ("cat"). The position vector says **where**
("second slot"). During training, the model learned to encode these two signals in non-overlapping
directions within the 768 dimensions, so they don't interfere with each other.

> **为什么直接相加就行?**词元向量表达**是什么**("cat"),位置向量表达**在哪**
> (“第二个槽位”)。训练过程中,模型学会了把这两种信号编码在 768 维中
> 互不重叠的方向上,因此互不干扰。

Analogy: a shipping label. The item description says what's in the box. The tracking number says
where it is in the delivery sequence. Both stamped on the same label — the warehouse reads whichever piece it needs.

> 类比:快递面单。物品描述写的是箱子里装的是什么;运单号标示它在配送序列中的哪一环。
> 两者印在同一张面单上——仓库需要哪段信息就去读哪段。

In [4]:
wpe = weights["wpe.weight"]  # Position embedding table(位置嵌入表)

print(f"Position embedding matrix (wpe.weight)")
print(f"  Shape: {wpe.shape}")
print(f"  Max sequence length: {wpe.shape[0]}")
print()

# Build the full input embeddings for "The cat sat"(构建 "The cat sat" 的完整输入嵌入)
token_ids = encode("The cat sat")
seq_len = len(token_ids)

token_embeddings = wte[token_ids]        # (seq_len, 768) — what each token is(每个词元是什么)
position_embeddings = wpe[:seq_len]       # (seq_len, 768) — where each token is(每个词元在哪)
input_embeddings = token_embeddings + position_embeddings  # element-wise add(逐元素相加)

print(f"Token embeddings shape:    {token_embeddings.shape}")
print(f"Position embeddings shape: {position_embeddings.shape}")
print(f"Input embeddings shape:    {input_embeddings.shape}")
print()
print("This is the actual input to the first transformer block.")
print("A matrix of shape (seq_len, 768) — one 768-dim vector per token.")

Position embedding matrix (wpe.weight)
  Shape: (1024, 768)
  Max sequence length: 1024

Token embeddings shape:    (3, 768)
Position embeddings shape: (3, 768)
Input embeddings shape:    (3, 768)

This is the actual input to the first transformer block.
A matrix of shape (seq_len, 768) — one 768-dim vector per token.


## Step 3: Do Embeddings Capture Meaning?(第三步:嵌入能捕捉语义吗?)

These 768-dimensional vectors aren't random — they were learned during training.  
Similar words should have similar vectors. Let's check using **cosine similarity**:

> 这些 768 维向量不是随机的——它们是训练学出来的。  
> 相似的词应该有相似的向量。我们用**余弦相似度(cosine similarity)**来验证:

$$\text{similarity}(a, b) = \frac{a \cdot b}{\|a\| \cdot \|b\|}$$

> (公式:相似度 = 两向量的点积 ÷ 两个向量模长的乘积)

Result is between -1 (opposite) and +1 (identical direction).  
Think of it like: two vectors pointing the same direction = similar meaning.

> 结果介于 -1(方向相反)到 +1(方向完全相同)之间。  
> 可以这样理解:两个向量指向同一方向 = 语义相似。

In [5]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def get_embedding(word):
    """Get the token embedding for a single-token word.(取单词对应的词元嵌入。)"""
    ids = encode(word)
    if len(ids) == 1:
        return wte[ids[0]]
    # For multi-token words, average the embeddings(被切成多个词元的词,对其嵌入取平均)
    return np.mean(wte[ids], axis=0)

# Compare word pairs(比较词对)
pairs = [
    (" king", " queen"),
    (" king", " throne"),
    (" king", " potato"),
    (" cat", " dog"),
    (" cat", " server"),
    (" fast", " quick"),
    (" fast", " slow"),
    (" GPU", " CPU"),
    (" GPU", " sandwich"),
]

print(f"{'Word A':<12s} {'Word B':<12s} {'Similarity':>10s}  Interpretation")
print("-" * 65)
for a, b in pairs:
    sim = cosine_similarity(get_embedding(a), get_embedding(b))
    bar = "#" * int(max(0, sim) * 30)
    label = "high" if sim > 0.5 else "medium" if sim > 0.2 else "low"
    print(f"{a:<12s} {b:<12s} {sim:>10.4f}  {bar} ({label})")

Word A       Word B       Similarity  Interpretation
-----------------------------------------------------------------
 king         queen           0.6573  ################### (high)
 king         throne          0.5060  ############### (high)
 king         potato          0.3010  ######### (medium)
 cat          dog             0.5498  ################ (high)
 cat          server          0.2269  ###### (medium)
 fast         quick           0.5772  ################# (high)
 fast         slow            0.5100  ############### (high)
 GPU          CPU             0.6756  #################### (high)
 GPU          sandwich        0.2397  ####### (medium)


## Step 4: Position Embeddings — Do Nearby Positions Look Similar?(第四步:位置嵌入——相邻位置更像吗?)

Intuition says positions close together should have more similar embeddings than positions far apart.  
Let's verify: how does similarity decay with distance?

> 直觉上,相邻位置的嵌入应该比相距很远的位置更相似。  
> 来验证一下:相似度随距离如何衰减?

In [6]:
# Compare position 0 to positions 1, 10, 100, 500, 1000(把位置 0 与位置 1、10、100、500、1000 对比)
pos0 = wpe[0]

print("Position embedding similarity (relative to position 0):")
print(f"{'Position':<12s} {'Distance':>10s} {'Cosine Sim':>12s}")
print("-" * 38)

for pos in [1, 2, 5, 10, 50, 100, 500, 1023]:
    sim = cosine_similarity(pos0, wpe[pos])
    bar = "#" * int(max(0, (sim + 1) / 2) * 20)  # normalize to 0-1 for display(归一化到 0-1 便于展示)
    print(f"  pos {pos:<6d} {pos:>10d} {sim:>12.4f}  {bar}")

print()
print("Nearby positions are more similar — the model can sense distance.")
print("This is how the model knows word order without explicit position markers.")

Position embedding similarity (relative to position 0):
Position       Distance   Cosine Sim
--------------------------------------
  pos 1               1       0.5242  ###############
  pos 2               2       0.5021  ###############
  pos 5               5       0.4890  ##############
  pos 10             10       0.4443  ##############
  pos 50             50       0.3645  #############
  pos 100           100       0.2991  ############
  pos 500           500       0.0612  ##########
  pos 1023         1023      -0.0485  #########

Nearby positions are more similar — the model can sense distance.
This is how the model knows word order without explicit position markers.


## Step 5: Putting It Together — The Full Embedding Pipeline(第五步:串起来——完整的嵌入流水线)

Here's the complete embedding function that will be the first step of our forward pass:

> 下面是完整的嵌入函数,它将作为我们前向传播的第一步:

```
"The cat sat on the mat"
         │
   encode()          (tokenizer)
         │
  [464, 9246, 3332, 319, 262, 2603]
         │
   wte[ids]          (token lookup — what each token is)
         │
   + wpe[:seq_len]   (position lookup — where each token is)
         │
   (6, 768) matrix   (ready for transformer blocks)
```

> (流程:输入文本 → encode() 分词 → 词元 ID 序列 → wte[ids] 查词元(是什么)→ 加上 wpe[:seq_len] 位置向量(在哪)→ 得到 (6, 768) 矩阵,交给 Transformer 块)

In [7]:
def embed(token_ids, wte, wpe):
    """Convert token IDs to input embeddings.(把词元 ID 转成输入嵌入。)
    
    Like DNS resolution: token ID -> vector the model can route.(类比 DNS 解析:词元 ID -> 模型可路由的向量)
    
    token_ids: list of integers
    wte: token embedding matrix [vocab_size, d_model]
    wpe: position embedding matrix [max_seq_len, d_model]
    returns: numpy array of shape (seq_len, d_model)
    """
    seq_len = len(token_ids)
    token_emb = wte[token_ids]      # (seq_len, 768)
    pos_emb = wpe[:seq_len]         # (seq_len, 768)
    return token_emb + pos_emb      # (seq_len, 768)

# Test it end-to-end(端到端测试)
text = "The cat sat on the mat"
token_ids = encode(text)
embeddings = embed(token_ids, wte, wpe)

print(f"Input text:     '{text}'")
print(f"Token IDs:      {token_ids}")
print(f"Embedding shape: {embeddings.shape}")
print()
print(f"Each row is a 768-dimensional vector.")
print(f"This {embeddings.shape[0]}x{embeddings.shape[1]} matrix is the input to transformer block h.0.")
print()
print(f"Memory for this input: {embeddings.nbytes} bytes ({embeddings.nbytes/1024:.1f} KB)")
print(f"Memory for the embedding tables: {(wte.nbytes + wpe.nbytes) / (1024*1024):.1f} MB")

Input text:     'The cat sat on the mat'
Token IDs:      [464, 3797, 3332, 319, 262, 2603]
Embedding shape: (6, 768)

Each row is a 768-dimensional vector.
This 6x768 matrix is the input to transformer block h.0.

Memory for this input: 18432 bytes (18.0 KB)
Memory for the embedding tables: 150.2 MB


## Key Takeaways(核心要点)

- **Token embedding is a table lookup**, not a matrix multiplication. `wte[token_id]` gives you the vector. Like an array index.
- **Position embedding adds location awareness**. Without it, the model can't tell word order.
- **Both tables are learned** during training — similar words end up with similar vectors.
- The input to the transformer is: `wte[token_ids] + wpe[:seq_len]` — a `(seq_len, 768)` matrix.
- **Embedding tables take memory**: GPT-2's wte is ~147 MB. Larger vocabs (128K+) in modern models take even more.

> - **词元嵌入是查表**,不是矩阵乘法。`wte[token_id]` 直接给出向量,就像数组下标。
> - **位置嵌入赋予位置感知**。没有它,模型分不清词序。
> - **两张表都是训练学出来的**——相似的词最终有相似的向量。
> - Transformer 的输入是:`wte[token_ids] + wpe[:seq_len]`——一个 `(seq_len, 768)` 矩阵。
> - **嵌入表占内存**:GPT-2 的 wte 约 147 MB;现代模型更大的词表(128K+)占得更多。

## Next(下一步)

**Notebook 04** — We take this `(seq_len, 768)` matrix and pass it through **attention** — the core operation where tokens look at each other.

> **Notebook 04**——我们把这个 `(seq_len, 768)` 矩阵送入**注意力(attention)**——词元之间相互“对视”的核心运算。

## References(参考资料)

- *Inference Engineering* Ch 2.2.2 (pp. 50–51) — Embedding layer, transformer block structure
- *Inference Engineering* Ch 2.2.1 (pp. 49–50) — Model architecture, config.json

> - 《Inference Engineering》第 2.2.2 章(第 50–51 页)——嵌入层、Transformer 块结构
> - 《Inference Engineering》第 2.2.1 章(第 49–50 页)——模型架构、config.json